# Phase 1: Generate Platinum Training Data

**NBME Pipeline — NBME Score Clinical Patient Notes**

This notebook pseudo-labels 10,000 unannotated patient notes using FAISS few-shot retrieval + Qwen3.5-9B-Instruct-AWQ via vLLM.

**Hardware**: Single T4 16 GB GPU (Colab Free Tier)  
**Runtime**: ~2–4 hours

**Output**: `augmented_train.csv`

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
!pip install -q \
  "vllm>=0.9.0" \
  "transformers>=5.5.0" \
  "sentence-transformers" \
  "faiss-gpu" \
  "rapidfuzz" \
  "pydantic" \
  "tqdm" \
  "pandas" \
  "numpy" \
  "xgrammar"

print("Installation complete ✓")

In [ ]:
# ── Hugging Face login (required to download Qwen model) ─────────────────────
# Run this cell and enter your HF token when prompted.
# Get your token at: https://huggingface.co/settings/tokens
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# ── Upload Kaggle competition data ───────────────────────────────────────────
# Option A: Upload files manually via Colab file browser (left panel → upload icon)
# Option B: Use kaggle API:
#   !pip install kaggle
#   # Upload kaggle.json to Colab, then:
#   !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#   !kaggle competitions download -c nbme-score-clinical-patient-notes
#   !unzip -q nbme-score-clinical-patient-notes.zip

# Verify files are present:
import os
for f in ['features.csv', 'patient_notes.csv', 'train.csv', 'test.csv']:
    status = "✓" if os.path.exists(f) else "✗ MISSING"
    print(f"  {f}: {status}")

## Script Configuration

Edit the `CONFIG` dictionary below to adjust paths, sampling size, and model parameters.

In [ ]:
#!/usr/bin/env python3
# =============================================================================
# IMPORTS
# =============================================================================
import ast
import gc
import json
import logging
import re
import sys
from pathlib import Path
from typing import Optional

import faiss
import numpy as np
import pandas as pd
import torch
from pydantic import BaseModel
from rapidfuzz import fuzz, process as rfprocess
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams

# =============================================================================
# GLOBAL CONFIGURATION
# =============================================================================
CONFIG = {
    # ── Paths ─────────────────────────────────────────────────────────────────
    "DATA_DIR":          Path("."),
    "OUTPUT_FILE":       Path("augmented_train.csv"),
    "FAISS_INDEX_FILE":  Path("faiss_features.index"),
    "FAISS_META_FILE":   Path("faiss_metadata.parquet"),

    # ── Data sampling ─────────────────────────────────────────────────────────
    "SAMPLE_SIZE":   10_000,
    "RANDOM_SEED":   42,

    # ── Embedding (FAISS) ─────────────────────────────────────────────────────
    "EMBED_MODEL":       "all-MiniLM-L6-v2",   # 80 MB, fast on CPU
    "EMBED_BATCH_SIZE":  64,
    "TOP_K_EXAMPLES":    3,                     # few-shot examples per query

    # ── vLLM / LLM ────────────────────────────────────────────────────────────
    "LLM_MODEL":          "Qwen/Qwen3.5-9B-Instruct-AWQ",
    "LLM_QUANTIZATION":   "awq",
    "LLM_DTYPE":          "float16",   # T4 has no BF16 hardware support
    "GPU_MEM_UTIL":       0.82,
    "MAX_MODEL_LEN":      4096,        # tokens; lower if OOM → try 2048
    "MAX_NEW_TOKENS":     300,
    "LLM_TEMPERATURE":    0.0,         # greedy → deterministic, best F1

    # ── Generation batching ───────────────────────────────────────────────────
    "GENERATION_BATCH_SIZE": 256,

    # ── Span-matching ─────────────────────────────────────────────────────────
    "FUZZY_SCORE_CUTOFF": 72,
}

# =============================================================================
# LOGGING
# =============================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)
print("Imports and configuration loaded ✓")

## SECTION 2 — FAISS VECTOR INDEX (FEW-SHOT RETRIEVAL)

In [ ]:
def build_faiss_index(
    train_df:    pd.DataFrame,
    pn_df:       pd.DataFrame,
    features_df: pd.DataFrame,
    cfg:         dict,
) -> tuple:
    """
    Build (or load from cache) a FAISS IndexFlatIP over train.csv examples.

    Each vector encodes  "Feature: <feature_text>  Annotation: <annotation>"
    so that a query on feature_text alone retrieves semantically similar
    annotation-style examples.

    Returns
    -------
    index    : faiss.IndexFlatIP
    metadata : list[dict]  — parallel to index vectors; keys: feature_text,
                             annotation, pn_history, location
    """
    idx_path  = cfg["FAISS_INDEX_FILE"]
    meta_path = cfg["FAISS_META_FILE"]

    # ── Load from disk if cache exists ────────────────────────────────────────
    if idx_path.exists() and meta_path.exists():
        log.info("Loading cached FAISS index …")
        index    = faiss.read_index(str(idx_path))
        metadata = pd.read_parquet(meta_path).to_dict("records")
        log.info(f"  Loaded {index.ntotal} vectors (dim={index.d})")
        return index, metadata

    log.info("Building FAISS index from train.csv …")

    # ── Build pn_num → pn_history lookup ─────────────────────────────────────
    pn_map   = pn_df.set_index("pn_num")["pn_history"].to_dict()
    feat_map = (
        features_df
        .set_index(["case_num", "feature_num"])["feature_text"]
        .to_dict()
    )

    embed_texts = []
    metadata    = []

    for _, row in train_df.iterrows():
        feature_text = feat_map.get((row["case_num"], row["feature_num"]), "")
        pn_history   = pn_map.get(row["pn_num"], "")

        # Flatten annotation list → single string
        annotations = row["annotation"]  # already a Python list
        annotation_str = " | ".join(
            a for a in annotations if isinstance(a, str) and a.strip()
        )

        # Skip rows with missing critical fields
        if not feature_text or not annotation_str:
            continue

        embed_texts.append(
            f"Feature: {feature_text}  Annotation: {annotation_str}"
        )
        metadata.append({
            "feature_text": feature_text,
            "annotation":   annotation_str,
            # Truncate note to 500 chars to keep metadata small
            "pn_history":   (pn_history or "")[:500],
            "location":     str(row["location"]),
        })

    log.info(f"  Building embeddings for {len(embed_texts)} examples …")
    embed_model = SentenceTransformer(cfg["EMBED_MODEL"])
    embeddings  = embed_model.encode(
        embed_texts,
        batch_size=cfg["EMBED_BATCH_SIZE"],
        show_progress_bar=True,
        normalize_embeddings=True,   # unit-norm → dot product = cosine similarity
        convert_to_numpy=True,
    ).astype(np.float32)

    # Free embedding model before building LLM (save VRAM)
    del embed_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ── Build FAISS IndexFlatIP ───────────────────────────────────────────────
    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    log.info(f"  FAISS index built: {index.ntotal} vectors, dim={dim}")

    # ── Persist to disk ───────────────────────────────────────────────────────
    faiss.write_index(index, str(idx_path))
    pd.DataFrame(metadata).to_parquet(meta_path, index=False)
    log.info("  FAISS index cached to disk.")

    return index, metadata


def retrieve_few_shot_examples(
    query_feature_text: str,
    index,                           # faiss.Index (typed loosely for stub compat)
    metadata:           list,
    embed_model,                     # SentenceTransformer
    top_k:              int = 3,
) -> list:
    """
    Retrieve top-k semantically similar train examples for a given feature_text.
    Used to build the few-shot context in each LLM prompt.
    """
    query_vec = embed_model.encode(
        [f"Feature: {query_feature_text}  Annotation:"],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    distances, indices = index.search(query_vec, top_k)
    return [metadata[i] for i in indices[0] if 0 <= i < len(metadata)]

## SECTION 3 — PROMPT CONSTRUCTION

In [ ]:
SYSTEM_PROMPT = (
    "You are a clinical NLP specialist. "
    "Given a patient note and a clinical feature, extract the EXACT verbatim text spans "
    "from the note that express that feature. "
    "Rules:\n"
    "  1. Copy text character-for-character — do NOT paraphrase.\n"
    "  2. If the feature is absent from the note, return an empty list.\n"
    "  3. Output ONLY valid JSON — no markdown, no explanation, no <think> blocks.\n"
    "Output format: {\"spans\": [\"exact text 1\", \"exact text 2\"]}"
)


def build_messages(
    feature_text:      str,
    pn_history:        str,
    few_shot_examples: list,
) -> list:
    """
    Construct a chat-message list for llm.chat().

    Uses Qwen-style <|im_start|> / <|im_end|> convention via vLLM's chat()
    method which automatically applies the tokenizer's chat template.

    Returns
    -------
    list[dict]  — [{"role": ..., "content": ...}, ...]
    """
    # ── Build few-shot block ──────────────────────────────────────────────────
    examples_block = ""
    for i, ex in enumerate(few_shot_examples, start=1):
        note_excerpt = ex["pn_history"][:300].replace("\n", " ").strip()
        # annotation is already a " | "-joined string
        ann_parts = [repr(a.strip()) for a in ex["annotation"].split(" | ") if a.strip()]
        ann_json  = "[" + ", ".join(ann_parts) + "]"
        examples_block += (
            f"\n[Example {i}]\n"
            f"Note (excerpt): \"{note_excerpt}\"\n"
            f"Feature: {ex['feature_text']}\n"
            f"Answer: {{\"spans\": {ann_json}}}\n"
        )

    # ── User message ──────────────────────────────────────────────────────────
    target_note = pn_history.replace("\n", " ").strip()
    user_content = (
        f"Here are labelled examples:{examples_block}\n"
        f"---\n"
        f"Now label this note.\n"
        f"Note: \"{target_note}\"\n"
        f"Feature: {feature_text}\n\n"
        # /no_think disables Qwen3.5's chain-of-thought mode for clean JSON output
        "/no_think"
    )

    return [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_content},
    ]

## SECTION 4 — SPAN → CHARACTER POSITION MAPPING

In [ ]:
def _strip_think_tokens(text: str) -> str:
    """Remove Qwen3.5 <think>...</think> blocks that may precede the JSON."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def find_span_locations(
    span_texts:    list,
    pn_history:    str,
    fuzzy_cutoff:  int = 72,
) -> list:
    """
    Map a list of extracted text spans to "start end" character-offset strings.

    Strategy (in order):
      1. Exact substring match                      → fastest, most accurate
      2. Case-insensitive exact match               → handles LLM capitalisation fixes
      3. rapidfuzz sliding-window (fixed win size)  → handles minor typos / spacing

    Only includes a span if it can be located with sufficient confidence.

    Parameters
    ----------
    span_texts   : list of verbatim strings from the LLM
    pn_history   : the full patient note text
    fuzzy_cutoff : minimum rapidfuzz ratio score (0–100) to accept a fuzzy hit

    Returns
    -------
    list of "start end" strings, parallel to accepted spans
    """
    if not span_texts or not pn_history:
        return []

    locations  = []
    pn_lower   = pn_history.lower()

    for span in span_texts:
        span = span.strip()
        if not span:
            continue

        # ── 1. Exact match ────────────────────────────────────────────────────
        idx = pn_history.find(span)
        if idx != -1:
            locations.append(f"{idx} {idx + len(span)}")
            continue

        # ── 2. Case-insensitive exact match ───────────────────────────────────
        span_lower = span.lower()
        idx = pn_lower.find(span_lower)
        if idx != -1:
            locations.append(f"{idx} {idx + len(span)}")
            continue

        # ── 3. Fuzzy sliding-window match ─────────────────────────────────────
        # We only scan windows within ±20 % of the span length to keep it O(n).
        # Notes are ≤ 950 chars so this loop is fast even in pure Python.
        span_len = len(span)
        min_win  = max(1, int(span_len * 0.80))
        max_win  = min(len(pn_history), int(span_len * 1.20))

        best_score = 0
        best_start = -1
        best_end   = -1

        for win_size in range(min_win, max_win + 1):
            n_windows = len(pn_history) - win_size + 1
            if n_windows <= 0:
                continue
            # Build candidate strings for this window size (one pass)
            candidates = [pn_history[s : s + win_size] for s in range(n_windows)]
            result = rfprocess.extractOne(
                span,
                candidates,
                scorer=fuzz.ratio,
                score_cutoff=fuzzy_cutoff,
            )
            if result is not None:
                _text, score, pos = result
                if score > best_score:
                    best_score = score
                    best_start = pos
                    best_end   = pos + win_size

        if best_score >= fuzzy_cutoff and best_start != -1:
            locations.append(f"{best_start} {best_end}")
        # If no match found above cutoff → span is skipped (unreliable)

    return locations

## SECTION 5 — LLM INITIALISATION

In [ ]:
class SpanOutput(BaseModel):
    """Pydantic schema — enforces JSON output structure via XGrammar."""
    spans: list[str]


def init_llm(cfg: dict) -> LLM:
    """
    Initialise vLLM with Qwen3.5-9B-Instruct-AWQ.

    Key flags for T4 16 GB:
      • quantization="awq"          → 4-bit AWQ keeps model in ~6-8 GB VRAM
      • dtype="float16"             → T4 lacks BF16 hardware support
      • gpu_memory_utilization=0.82 → reserves ~3 GB for KV cache + activations
      • guided_decoding_backend     → XGrammar (faster than Outlines for batches)
      • enforce_eager=False         → allow CUDA graphs for throughput
    """
    if not torch.cuda.is_available():
        log.warning("CUDA not available — inference will be extremely slow on CPU.")

    log.info(f"Initialising vLLM  model={cfg['LLM_MODEL']} …")
    llm = LLM(
        model                    = cfg["LLM_MODEL"],
        quantization             = cfg["LLM_QUANTIZATION"],
        dtype                    = cfg["LLM_DTYPE"],
        gpu_memory_utilization   = cfg["GPU_MEM_UTIL"],
        max_model_len            = cfg["MAX_MODEL_LEN"],
        trust_remote_code        = True,
        enforce_eager            = False,
        guided_decoding_backend  = "xgrammar",   # XGrammar is default in vLLM ≥0.12
        seed                     = cfg["RANDOM_SEED"],
    )
    log.info("vLLM initialised successfully.")
    return llm


def make_sampling_params(cfg: dict) -> SamplingParams:
    """
    SamplingParams with XGrammar-enforced JSON schema.
    temperature=0 → greedy decoding for maximum label consistency.
    """
    return SamplingParams(
        temperature     = cfg["LLM_TEMPERATURE"],
        max_tokens      = cfg["MAX_NEW_TOKENS"],
        guided_decoding = GuidedDecodingParams(
            json    = SpanOutput.model_json_schema(),
            backend = "xgrammar",
        ),
    )

## SECTION 6 — MAIN GENERATION LOOP

In [ ]:
def generate_pseudo_labels(
    sample_notes:   pd.DataFrame,
    train_df:       pd.DataFrame,
    features_df:    pd.DataFrame,
    faiss_index,                     # faiss.Index
    faiss_metadata: list,
    llm,                             # vllm.LLM
    sampling_params,                 # vllm.SamplingParams
    cfg:            dict,
) -> pd.DataFrame:
    """
    Full pipeline: build prompts → batch LLM inference → parse JSON → map spans.

    Returns
    -------
    aug_df : pd.DataFrame  — columns: id, pn_num, feature_num, case_num,
                                            annotation, location
                  matching the exact schema of train.csv
    """
    # ── Build lookup tables ───────────────────────────────────────────────────
    feat_map = (
        features_df
        .set_index(["case_num", "feature_num"])["feature_text"]
        .to_dict()
    )
    # case_num → list[feature_num]
    case_features: dict = (
        features_df
        .groupby("case_num")["feature_num"]
        .apply(list)
        .to_dict()
    )

    # ── Load embedding model for per-feature FAISS queries ────────────────────
    log.info("Loading sentence-transformer for FAISS retrieval …")
    embed_model = SentenceTransformer(cfg["EMBED_MODEL"])

    # =========================================================================
    # PHASE A — Build all (note × feature) prompts
    # =========================================================================
    log.info("Building all (note × feature) prompts …")

    all_messages: list = []   # each element: list[dict]  (one conversation)
    all_meta:     list = []   # parallel metadata

    for _, note_row in tqdm(
        sample_notes.iterrows(),
        total=len(sample_notes),
        desc="Building prompts",
    ):
        pn_num     = int(note_row["pn_num"])
        case_num   = int(note_row["case_num"])
        pn_history = note_row["pn_history"]

        # Guard: skip empty notes
        if not isinstance(pn_history, str) or not pn_history.strip():
            continue

        feature_nums = case_features.get(case_num, [])

        for feature_num in feature_nums:
            feature_text = feat_map.get((case_num, feature_num), "")
            if not feature_text:
                continue

            # Retrieve 3 semantically similar train examples for this feature
            few_shot = retrieve_few_shot_examples(
                query_feature_text = feature_text,
                index              = faiss_index,
                metadata           = faiss_metadata,
                embed_model        = embed_model,
                top_k              = cfg["TOP_K_EXAMPLES"],
            )

            messages = build_messages(
                feature_text      = feature_text,
                pn_history        = pn_history,
                few_shot_examples = few_shot,
            )
            all_messages.append(messages)
            all_meta.append({
                "pn_num":       pn_num,
                "case_num":     case_num,
                "feature_num":  feature_num,
                "feature_text": feature_text,
                "pn_history":   pn_history,
            })

    log.info(f"Total (note × feature) pairs to label: {len(all_messages)}")

    # Free embedding model — we no longer need it; reclaim VRAM for vLLM KV cache
    del embed_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # =========================================================================
    # PHASE B — Batch LLM inference
    # =========================================================================
    log.info("Running batch LLM inference …")

    all_raw_outputs: list = []
    batch_size = cfg["GENERATION_BATCH_SIZE"]
    n_batches  = (len(all_messages) + batch_size - 1) // batch_size

    for b_idx in tqdm(range(n_batches), desc="LLM batches"):
        start = b_idx * batch_size
        end   = min(start + batch_size, len(all_messages))
        batch = all_messages[start:end]

        try:
            # llm.chat() applies the tokenizer's chat template automatically.
            outputs = llm.chat(batch, sampling_params=sampling_params)
            all_raw_outputs.extend(outputs)
        except Exception as exc:
            log.error(f"Batch {b_idx} failed: {exc}")
            # Pad with None so metadata stays aligned
            all_raw_outputs.extend([None] * len(batch))

        # Periodic VRAM cleanup (KV cache can accumulate between batches)
        if (b_idx + 1) % 10 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # =========================================================================
    # PHASE C — Parse JSON outputs + map spans to character positions
    # =========================================================================
    log.info("Parsing outputs and mapping spans to character offsets …")

    aug_rows = []

    for output, meta in tqdm(
        zip(all_raw_outputs, all_meta),
        total=len(all_raw_outputs),
        desc="Span mapping",
    ):
        pn_history   = meta["pn_history"]
        pn_num       = meta["pn_num"]
        case_num     = meta["case_num"]
        feature_num  = meta["feature_num"]

        # ── Unique ID matching train.csv format: "{pn_num:05d}_{feature_num:03d}" ──
        row_id = f"{pn_num:05d}_{feature_num:03d}"

        # ── Parse JSON from LLM output ────────────────────────────────────────
        spans: list = []
        if output is not None:
            try:
                raw_text = output.outputs[0].text.strip()
                # Strip any <think>…</think> blocks Qwen3.5 might emit
                raw_text = _strip_think_tokens(raw_text)
                parsed   = json.loads(raw_text)
                spans    = [
                    s for s in parsed.get("spans", [])
                    if isinstance(s, str) and s.strip()
                ]
            except Exception as exc:
                raw_preview = (
                    output.outputs[0].text[:120] if output else "None"
                )
                log.debug(f"JSON parse error for {row_id}: {exc} | raw='{raw_preview}'")
                spans = []

        # ── Map spans to character-level locations ────────────────────────────
        locations = find_span_locations(
            span_texts   = spans,
            pn_history   = pn_history,
            fuzzy_cutoff = cfg["FUZZY_SCORE_CUTOFF"],
        )

        # ── Format to match train.csv schema exactly ──────────────────────────
        # annotation: Python list repr of extracted texts  → e.g. "['text1', 'text2']"
        # location:   Python list repr of "start end" str  → e.g. "['10 25', '40 50']"
        #
        # If nothing was found, store [""] to match the train.csv convention
        # (where unannotated rows have annotation=[""] location=[""]).
        annotation_col = spans     if spans     else [""]
        location_col   = locations if locations else [""]

        aug_rows.append({
            "id":          row_id,
            "pn_num":      pn_num,
            "feature_num": feature_num,
            "case_num":    case_num,
            "annotation":  str(annotation_col),
            "location":    str(location_col),
        })

    aug_df = pd.DataFrame(aug_rows)

    # ── Summary stats ─────────────────────────────────────────────────────────
    total      = len(aug_df)
    non_empty  = (aug_df["location"] != str([""])).sum()
    fill_rate  = 100.0 * non_empty / max(total, 1)
    log.info(
        f"Generated {total} pseudo-labelled rows | "
        f"non-empty labels: {non_empty} ({fill_rate:.1f} %)"
    )

    return aug_df


# =============================================================================
# MAIN
# =============================================================================

def main():
    cfg = CONFIG
    log.info("=" * 65)
    log.info("  PHASE 1: Pseudo-Label Generation")
    log.info("=" * 65)

    # ── 1. Load & filter data ─────────────────────────────────────────────────
    sample_notes, train_df, features_df, pn_df = load_and_filter_data(cfg)

    # ── 2. Build FAISS index (cached after first run) ─────────────────────────
    faiss_index, faiss_metadata = build_faiss_index(
        train_df    = train_df,
        pn_df       = pn_df,
        features_df = features_df,
        cfg         = cfg,
    )

    # ── 3. Initialise vLLM ───────────────────────────────────────────────────
    llm             = init_llm(cfg)
    sampling_params = make_sampling_params(cfg)

    # ── 4. Generate pseudo-labels ────────────────────────────────────────────
    aug_df = generate_pseudo_labels(
        sample_notes    = sample_notes,
        train_df        = train_df,
        features_df     = features_df,
        faiss_index     = faiss_index,
        faiss_metadata  = faiss_metadata,
        llm             = llm,
        sampling_params = sampling_params,
        cfg             = cfg,
    )

    # ── 5. Save output ───────────────────────────────────────────────────────
    out_path = cfg["OUTPUT_FILE"]
    aug_df.to_csv(out_path, index=False)
    log.info(f"Augmented dataset saved → {out_path}  shape={aug_df.shape}")
    log.info(f"Sample rows:\n{aug_df.head(5).to_string()}")

    # ── Final VRAM cleanup ───────────────────────────────────────────────────
    del llm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    log.info("Phase 1 complete ✓")


if __name__ == "__main__":
    main()

## Run Phase 1

In [ ]:
# =============================================================================
# MAIN
# =============================================================================

def main():
    cfg = CONFIG
    log.info("=" * 65)
    log.info("  PHASE 1: Pseudo-Label Generation")
    log.info("=" * 65)

    # ── 1. Load & filter data ─────────────────────────────────────────────────
    sample_notes, train_df, features_df, pn_df = load_and_filter_data(cfg)

    # ── 2. Build FAISS index (cached after first run) ─────────────────────────
    faiss_index, faiss_metadata = build_faiss_index(
        train_df    = train_df,
        pn_df       = pn_df,
        features_df = features_df,
        cfg         = cfg,
    )

    # ── 3. Initialise vLLM ───────────────────────────────────────────────────
    llm             = init_llm(cfg)
    sampling_params = make_sampling_params(cfg)

    # ── 4. Generate pseudo-labels ────────────────────────────────────────────
    aug_df = generate_pseudo_labels(
        sample_notes    = sample_notes,
        train_df        = train_df,
        features_df     = features_df,
        faiss_index     = faiss_index,
        faiss_metadata  = faiss_metadata,
        llm             = llm,
        sampling_params = sampling_params,
        cfg             = cfg,
    )

    # ── 5. Save output ───────────────────────────────────────────────────────
    out_path = cfg["OUTPUT_FILE"]
    aug_df.to_csv(out_path, index=False)
    log.info(f"Augmented dataset saved → {out_path}  shape={aug_df.shape}")
    log.info(f"Sample rows:\n{aug_df.head(5).to_string()}")

    # ── Final VRAM cleanup ───────────────────────────────────────────────────
    del llm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    log.info("Phase 1 complete ✓")


if __name__ == "__main__":
    main()

main()